# Target Grouping and Identification

This notebook is intended to plan for Target identification and grouping.
Identification is based on a combination of DICOM Structure Type: 
*RT ROI Interpreted Type (3006,00A4)*
and the Structure ID:
*ROI Name (3006,0026)*

- The RT ROI Interpreted Type values that are defined as Targets are specified.

- Regular expressions are defined that can be used to identify target structures and to group them.

Test examples are Use to demonstrate the functionality of the identification and grouping process.

## Setup

### Imports

In [1]:
from typing import Dict, List, Union

from pathlib import Path
import re

from pprint import pprint

In [2]:
import xml.etree.ElementTree as ET
from itertools import chain


In [3]:
import pandas as pd
import xlwings as xw

In [4]:
from structure_set import StructureSet
from dicom import DicomStructureFile

INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Paths

In [5]:
base_path = Path.cwd()
dicom_path = base_path / "tests"
structure_filter_def = base_path / "src" / "webapp" / "config" / "structure_filter_rules.json"

## Load an example H&N DICOM structure set

### Path to the example DICOM file

In [6]:
dcm_file_path = dicom_path / "RS.HN_Struct.OROP.dcm"
#dcm_file_path = dicom_path / 'RS.CNS_FSRT_2_GTV.BRAI.dcm'
#dcm_file_path = dicom_path / 'RS.4Field_Logic.BRER.dcm'
#dcm_file_path = dicom_path / 'RS.LUNR_Test.LUNR.dcm'
#dcm_file_path = dicom_path / 'RS_4_field.dcm'
#dcm_file_path = dicom_path / 'RS.Pros_Equal.dcm'
#dcm_file_path = dicom_path / 'RS.PROS_Test.dcm'

### Load the DICOM file

In [7]:
dicom_file = DicomStructureFile(top_dir=dicom_path, file_path=dcm_file_path)
pprint(dicom_file.structure_set_info)

INFO:dicom:Successfully loaded DICOM dataset from RS.HN_Struct.OROP.dcm
INFO:dicom:Extracted 3666 contours from 61 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.HN_Struct.OROP.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.070 cm/pixel


{'File': WindowsPath("d:/OneDrive - Queen's University/Python/Projects/StructureRelations/tests/RS.HN_Struct.OROP.dcm"),
 'PatientID': 'HN_Struct',
 'PatientLastName': 'Head',
 'PatientName': 'Head^and^Neck^Structures',
 'SeriesDescription': 'ARIA RadOnc Structure Sets',
 'SeriesNumber': '234',
 'StructureSet': 'OROP',
 'StudyID': ''}


### Filter unwanted structures

In [8]:
filter_report = dicom_file.evaluate_structure_filters(structure_filter_def)
selection = filter_report.SelectedByDefault & filter_report.DisplayedByDefault
meta_data = dicom_file.get_structure_filter_metadata()[selection]
include_structures = list(meta_data['Structure ID'])

### No filtering Option

In [9]:
meta_data = dicom_file.get_structure_filter_metadata()
include_structures = list(meta_data['Structure ID'])

### List of structures that are included in the analysis

In [10]:
include_structures

['BODY',
 'DPV',
 'Avoid Post',
 'Brain',
 'BrainStem',
 'Cochlea L',
 'Cochlea R',
 'CTV 56 L',
 'CTV 56 R',
 'CTV 70',
 'Esophagus',
 'eval PTV 56',
 'eval PTV 70',
 'Globe L',
 'Globe R',
 'GTV',
 'GTVn',
 'Larynx',
 'Lens L',
 'Lens R',
 'Mandible',
 'opt Larynx',
 'opt Parotid L',
 'opt Parotid R',
 'opt PTV 56 L a',
 'opt PTV 56 L b',
 'opt PTV 56 R a',
 'opt PTV 56 R b',
 'opt PTV 70',
 'OpticChiasm',
 'OpticNerve L',
 'OpticNerve R',
 'Parotid B',
 'Parotid L',
 'Parotid R',
 'PRV BR + op',
 'PRV5 BrainStem',
 'PRV5 Optics',
 'PRV5 SpinalCanal',
 'PTV 56 L',
 'PTV 56 R',
 'PTV 70',
 'SpinalCanal',
 'Submandibular B',
 'Submandibular L',
 'Submandibular R',
 'Z2-all ptv',
 'BrachialPlexusL',
 'BrachialPlexusR',
 'Lip',
 'Z$LN_Neck_2347 L',
 'Z$LN_NECK_R',
 'Z$LN_Neck_V_L',
 'Musc_Sclmast_L',
 'Musc_Sclmast_R',
 'OralCavity',
 'Z9-real body',
 '1.0 cm Bolus',
 'Z7-ptv 70 crop',
 'D5320[cGy]-bob',
 'D5320[cGy]2']

### Determine the relationships between the structures

## Target Nomenclature

In [11]:
# Initialize dictionary with descriptions for each regex group.
group_definitions = {}
# Initialize list to hold regex sections.
regex_sections = []

### Target Modifier Prefixes
- `eval` Target volume explicitly intended for DVH evaluation
- `opt` Target volume only intended for optimization. 
- `mod` Target volume modified for unspecified reasons.
- `shell` Spherical shell surrounding a target volume used for optimizing dose fall-off. 

In [12]:
target_mod_def = {
    'eval': 'Target volume explicitly intended for DVH evaluation',
    'opt': 'Target volume only intended for optimization',
    'mod': 'Target volume modified for unspecified reasons',
    'shell': 'Spherical shell surrounding a target volume used for optimizing dose fall-off'
    }
# Add the target type definitions to the group_definitions dictionary.
group_definitions['Mod'] = target_mod_def

target_mod = '|'.join(target_mod_def.keys())
target_mod_pat = ''.join([
    r'(?P<Mod>',      # Start of *Mod* group
    f'{target_mod}',    # contains '|' separated items from target_mod_def.
    r')?',            # End of optional *Mod* group
    r'[ _]*'          # Optional space or '_'
    ])

regex_sections.append(target_mod_pat)

### Target Type

- The first set of characters must be one of the allowed target types:
    - GTV
    - CTV
    - ITV
    - IGTV (Internal Gross Target Volume—gross disease with margin for motion)
    - ICTV (Internal Clinical Target Volume—clinical disease with margin for motion)
    - PTV
    - PTV! for low-dose PTV volumes that exclude overlapping high-dose volumes

- Additional Target types not defined in TG-263:
  - Nodes, Node:  Used to identify a nodal volume that is not a GTV, but used to define a CTV or   PTV.
  - Iliac Vessels Special case of pelvic vessels used as a surrogate for pelvic nodes.
  - Edema: Used as a contrasted-based indicator of CTV volumes in CNS tumours.
  - Cavity: Used in breast cancer, where the GTV is defined as the surgical cavity.
  - Operative Bed: Used for post-surgery cases where the target is the operative bed.
  - HTV: High-risk target volume.
  - HRV: High-risk volume.

In [13]:
target_type_def = {
    'GTV': 'Gross Target Volume',
    'CTV': 'Clinical Target Volume',
    'PTV': 'Planning Target Volume',
    'ITV': 'Internal Target Volume',
    'IGTV': 'Internal Gross Target Volume',
    'ICTV': 'Internal Clinical Target Volume',
    'PTV!': 'Partial Planning Target Volume',
    # Additional local target types not defined in TG-263:
    'Nodes': 'Nodal Volume',
    'Node': 'Nodal Volume',
    'LN': 'Nodal Volume',
    'Iliac Vessels': 'Nodal Volume',
    'Edema': 'CTV Surrogate',
    'Cavity': 'GTV Surrogate',
    'Operative Bed': 'GTV Surrogate',
    'HTV': 'Clinical Target Volume',
    'HRV': 'Clinical Target Volume',
    }

# Add the target type definitions to the group_definitions dictionary.
group_definitions['TargetType'] = target_type_def

# Create a regular expression pattern to match the target types defined in target_type_def.
target_type = '|'.join(target_type_def.keys())
target_type_pat = ''.join([
    r'(?P<TargetType>',  # Start of required *TargetType* group
    f'{target_type}',    # contains '|' separated items from target_type.
    r')'                 # End of *TargetType* group.
    ])

regex_sections.append(target_type_pat)

### Target Classifier

- If used, the target classifier is placed after the target type with no spaces.
    - Allowed target classifiers are listed below: 
    - n: nodal (e.g., PTVn) 
    - p: primary (e.g., GTVp) 
    - m: metastatic (e.g., CTVm) *Non TG-263 target classifier*
    - sb: surgical bed (e.g., CTVsb) 
    - par: parenchyma (e.g., GTVpar) 
    - v:venous thrombosis (e.g., CTVv) 
    - vas: vascular (e.g., CTVvas)

- Other non TG-263 target classifiers are also used:
    - low: Low Dose Target
    - int: Intermediate Dose Target
    - IR: Intermediate Risk Target Volume
    - HR: High Risk Target Volume
    - PREOP: Pre-operative Target Volume
    - RES: Residual Disease Target Volume
    - Cavity: Target Expansion on the Cavity

In [14]:
target_classifier_def = {
    'n': 'nodal',
    'p': 'primary',
    'm': 'metastatic',
    'sb': 'surgical bed',
    'par': 'parenchyma',
    'v': 'venous thrombosis',
    'vas': 'vascular',
    'low': 'Low Dose Target',
    'int': 'Intermediate Dose Target',
    'IR': 'Intermediate Risk Target Volume',
    'HR': 'High Risk Target Volume',
    'PREOP': 'Pre-operative Target Volume',
    'RES': 'Residual Disease Target Volume',
    'Cavity': 'Target Expansion on the Cavity'
    }

# Add the target type definitions to the group_definitions dictionary.
group_definitions['Classifier'] = target_classifier_def

# Create a regular expression pattern to match the target types defined in target_classifier_def.
target_classifier = '|'.join(target_classifier_def.keys())
# Checks on characters following the classifier to ensure that it is not part
# of a longer word.  For example, 'n' should not match 'node'.
target_classifier_pat = ''.join([
    r'(?:'                   # Start of an optional non-capturing group.
    r'(?P<Classifier>',        # Start of required *Classifier* group
    f'{target_classifier}',      # contains '|' separated items from target_type.
    r')'                      # End of the optional *Classifier* group.
    r'(?![D-Zd-z]{{2}})'      # NOT followed by any two characters except for a, b, or c.
    r')?'                 # End of the optional non-capturing group.
    ])

regex_sections.append(target_classifier_pat)

### Target Number

- For multiple spatially distinct targets Arabic numerals are used after the 
  target type and classifier (e.g., PTV1, PTV2, GTVp1, GTVp2).

The regular expression is more complex than most to ensure that the target 
number is not part of a larger number, such as a dose value.

In [15]:
# Create a regular expression pattern to match the target number as 1 or 2 digits.
# A group definition dictionary is not required for the target number because
# it is just a number.

# Characters preceding and following the group also need to be tested to ensure
# that it is a target number and not a dose value or expansion number. The target
# number is not allowed to be proceeded by a digit, '.', '+', or '-'. It is also
# not allowed to be followed by a digit, '.', 'D', 'c', or 'm'.

target_number_pat = ''.join([
    r'(?:',                # Start of an optional non-capturing group.
    r'[ _]*',                # Optional space or '_'
    r'(?<![0-9.+-])',        # NOT preceded by another digit, '.', '+', or '-'.
    r'(?P<TargetNumber>',    # Start of *TargetNumber* group
    r'[1-9]',                  # A single digit.
    r'|',                      # OR
    r'1[0-5]',                 # A '1' and an additional digit between 0 and 4.
    r')'                     # End of the *TargetNumber* group.
    r'(?![0-9.Dcm])',        # NOT followed by a digit, '.', 'D', 'c', or 'm'.
    r')?',                   # End of the optional non-capturing group.
    ])

regex_sections.append(target_number_pat)

### Image Modality Designator

- Imaging modality follows the type/classifier/enumerator with an underscore 
  and then the image modality type (CT, PT, MR, SP)

- 'T' is also included as an abbreviation for MR T1, or 'T2'.  This is not a TG-263 designation.

- Image sequence order is indicated by a number immediately following the 
  image modality.

- Unlike the TG-263 protocol, multiple modalities cannot be included. 

In [16]:
target_modality_def = {
    'CT': 'CT',
    'PT': 'PET',
    'PET': 'PET',
    'MR': 'MRI',
    'MRI': 'MRI',
    'T': 'MRI',
    'US': 'Ultrasound',
    'SP': 'SPECT'
    }

# Add the target modality definitions to the group_definitions dictionary.
group_definitions['Modality'] = target_classifier_def

# Create a regular expression pattern to match the target types defined in target_modality_def.
target_modality = '|'.join(target_modality_def.keys())
target_modality_pat = ''.join([
    r'[ _]*',               # Optional space or '_'
    r'(?:',                 # Start of an optional non-capturing group.
    r'(?P<Modality>',         # Start of named group *Modality*
    f'{target_modality}',       # contains '|' separated items from target_modality.
    r')',                     # End of named group *Modality*
    r'(?P<ModalityNumber>',   # Start of optional named group *ModalityNumber*
    r'[0-9]{0,2}',              # Optional sequence number as 1 or 2 digits
    r')?',                    # End of optional named group *ModalityNumber*
    r')?'                   # End of optional group
    ])

regex_sections.append(target_modality_pat)

### Motion Management Designator

For 4D scanning target modifiers are available to indicate both aggregates and phase bins

- Aggregated *4D* target motion includes the following designators:
  - **MIP**: Maximum intensity projection, 
  - **AIP**: or *AVE*: Average intensity projection,
  - **MinIP**:  Minimum intensity projection,
  - **4D##**: Target volume on a particular motion phase, 
      where ## is the phase number (usually 10, 20, ...)


In [17]:
target_motion_def = {
    'MIP': 'Maximum intensity projection',
    'AIP': 'Average intensity projection',
    'AVE': 'Average intensity projection',
    'MinIP': 'Minimum intensity projection'
    }

# Add the target motion definitions to the group_definitions dictionary.
group_definitions['Motion'] = target_motion_def

# Create a regular expression pattern to match the target motion options.
# The named motion option and the 4D phase groups are supplied as seperate
# optional groups for simplicity.  Based on the regex expression, it as possible
# for both an aggregate and a phase indicator to coexist e.g. 'PTV Ave 4D10'.
# However this is not a logical combination and we assume that no one would
# actually combine them in a target name.
target_motion = '|'.join(target_motion_def.keys())
target_motion_pat = ''.join([
    r'[ _]*',             # Optional space or '_'
    r'(?P<Motion>',         # Start of optional named group *Motion*
    f'{target_motion}',     # contains '|' separated items from target_motion.
    r')?',                # End of optional named group *Motion*
    r'(?P<MotionPhase>',  # Start of optional named group *MotionPhase*
    r'4D[0-9][0-9]?'        # '4D' followed by one or two digits.
    r')?',                # End of optional named group *MotionPhase*
    ])

regex_sections.append(target_motion_pat)

### Nodal Region Indicator

For Nodal targets the name of the nodal chain is usually included.  
The nodal region sometimes has a part qualifier such as 'Nodes Axilla II', or 
LN_Neck_V', The qualifier is usually in roman numerals, but it can also be 
lowercase letters of arabic numbers. 

The following nodal regions are currently recognized:
- Int iliac: Internal Iliac Nodes
- ext iliac: External Iliac Nodes
- com iliac: Common Iliac Nodes
- Sacral: Sacral Nodes
- Axilla: Axillary Nodes
- IMC: Internal Mammary Chain Nodes
- SC: Supraclavicular Nodes
- Pyloric: Pyloric Nodes
- Subpyloric: Subpyloric Nodes
- Gastric: Gastric Nodes
- Splenic: Splenic Nodes
- Celiac: Celiac Nodes
- Pancreatic: Pancreatic Nodes
- Hepatoduod: Hepatoduodenal Nodes
- Hepatogastr: Hepatogastric Nodes
- Hepatic: Hepatic Nodes
- Para-Aortic: Para-Aortic Nodes
- Obturator: Obturator Nodes

In [18]:
# Region (The name of a nodal group region.)
nodal_region_def = {
    'Int iliac': 'Internal Iliac Nodes',
    'ext iliac': 'External Iliac Nodes',
    'com iliac': 'Common Iliac Nodes',
    'Sacral': 'Sacral Nodes',
    'Axilla': 'Axillary Nodes',
    'IMC': 'Internal Mammary Chain Nodes',
    'SC': 'Supraclavicular Nodes',
    'Pyloric': 'Pyloric Nodes',
    'Subpyloric': 'Subpyloric Nodes',
    'Gastric': 'Gastric Nodes',
    'Splenic': 'Splenic Nodes',
    'Celiac': 'Celiac Nodes',
    'Pancreatic': 'Pancreatic Nodes',
    'Hepatoduod': 'Hepatoduodenal Nodes',
    'Hepatogastr': 'Hepatogastric Nodes',
    'Hepatic': 'Hepatic Nodes',
    'Para-Aortic': 'Para-Aortic Nodes',
    'Obturator': 'Obturator Nodes',
    }

# Add the target motion definitions to the group_definitions dictionary.
group_definitions['Nodes'] = nodal_region_def


nodal_region = '|'.join(nodal_region_def.keys())
nodal_region_pat = ''.join([
    r'[ _]*',                 # Optional space or '_'
    r'(?:',                   # Start of an optional non-capturing group.
    r'(?P<Nodes>',              # Start of named group *Nodes*
    f'{nodal_region}',            # contains '|' separated items from nodal_region.
    r')',                       # End of named group *Nodes*
    r'[ _]*',                   # Optional space or '_'
    r'(?P<NodalSubSection>',    # Start of optional named group *NodalSubSection*
    r'[IVabc1-9]*'                # A combination of I, V, a,b,c, numbers 1 to 9.
    r')?',                      # End of optional named group *NodalSubSection*
    r')?',                    # End of optional non-capturing group.
   ])

regex_sections.append(nodal_region_pat)


### Optional Dose Specifier

- An optional dose specifier can be included.
- The dose specifier is preceded by an optional delimiter which can be:
   - A space,
   - A hyphen, (because dose is never subtracted)
   - An underscore character.

- Dose specifier can be one of:
   - Numeric dose
   - Dose per Fraction and number of Fractions
   - Relative Dose Level


#### Numeric Dose
- Units of cGy are recommended for numeric dose values. (e.g., PTV_5040).
- When specified in units of Gy, then ‘Gy’ should be appended to the numeric 
  value of the dose (e.g., PTV_50.4Gy). 
- For systems that do not allow use of a period, the ‘p’ character should be 
  substituted (e.g., PTV_50p4Gy)

#### Number of Fractions
- If the dose indicated must reflect the number of fractions used to reach the 
  total dose, there ar two ways to specify this:
    - Total dose, followed by / or 'in', followed by an integer number of 
       fractions. e.g. 60Gy/30 or 60Gy in 30
    - Dose per fraction, followed by 'x', followed by an integer number of 
       fractions. e.g. 2000cGyx3 or 20Gyx3


The overall numeric group contains three subsections:
1. the required numeric dose value, which can be an integer or a decimal number.
2. The optional dose units, which can be either cGy or Gy. 
3. The optional number of fractions, which contains two named groups,
    1. FractionDelimiter
    2. NumberOfFractions
The FractionDelimiter needs to be captured to know whether the dose is being specified a total dose or dose per fraction.

**Note:**
Surrounding the entire numeric dose group with parentheses is necessary to
distinguish between optional parts of the numeric dose specifier and required 
parts, while still designating the entire numeric dose as an optional component
of the overall target name.  Likewise, the parentheses around the optional number of fractions is necessary to indicate that while number of fractions is optional, if it is included both the delimiter and fractions components must be present.


In [19]:
numeric_dose_pat = ''.join([
    r'(?:'                      # Start of an optional non-capturing NumericDose group.
    # Required numeric dose value
    r'(?P<TargetDose>',           # Start of named group *TargetDose* (required for any numeric dose)
    r'[0-9]+',                      # Number before decimal place
    r'[.p]?',                       # '.' or 'p' as optional decimal place
    r'[0-9]*',                      # Optional Number after decimal place
    r')'                          # End of the *TargetDose* group

    # Optional space or '_'
    r'[ _]*'                      # Optional space or '_' between numeric dose and units

    # Optional dose units
   r'(?P<DoseUnits>',             # Start of optional named group *DoseUnits*
    r'[cGy]{2,3}',                  # Optional units: cGy or Gy.
    r')?'                         # End of optional named group *DoseUnits*

    # Optional space or '_'
    r'[ _]*'                      # Optional space or '_' between dose and fractions

   # Optional number of fractions
    r'(?:'                        # Start of an optional non-capturing number of fractions group.
    r'(?P<FractionDelimiter>',      # Start of named group *FractionDelimiter* (required part of fractions)
    r'/|in|x'                         # One of  either '/', 'in', 'x'
    r')'                            # End of the *FractionDelimiter* group
    r'(?P<Fractions>',              # Start of named group *Fractions* (required part of fractions)
    r'[0-9]+',                        # Number of fractions
    r')'                            # End of the *Fractions* group
    r')?'                         # End of the optional non-capturing number of fractions group

    r')?'                       # End of the optional non-capturing NumericDose group
    ])

regex_sections.append(numeric_dose_pat)

#### Relative Dose
- Relative dose is one of Low Medium or High:
    - High (e.g., PTV_High, CTV_High, GTV_High)
    - Mid: (e.g., PTV_Mid, CTV_Mid, GTV_Mid)
    - Low (e.g., PTV_Low, CTV_Low, GTV_Low) ◦ 

- Mid+2-digit enumerator: allows specification of more than three relative 
  dose levels (e.g., PTV_Low, PTV_Mid01, PTV_Mid02, PTV_Mid03, PTV_High). 
- Lower numbers correspond to lower dose values.

In [20]:
relative_dose_def = {
    'High': 'Highest target dose level',
    'Mid': 'Intermediate target dose level',
    'Low': 'Lowest target dose level'
    }

# Add the target motion definitions to the group_definitions dictionary.
group_definitions['RelativeDose'] = relative_dose_def

relative_dose = '|'.join(relative_dose_def.keys())
relative_dose_pat = ''.join([
    r'[ _]*',                   # Optional space or '_'
    r'(?:'                      # Start of an optional non-capturing RelativeDose group.
    r'(?P<RelativeDose>',         # Start of named group *RelativeDose*
    f'{relative_dose}',             # contains '|' separated items from relative_dose.
    r')',                         # End of named group *RelativeDose*
    r'(?P<RelativeDoseLevel>',    # Start of optional named group *RelativeDoseLevel*
    r'[0-9]{2}',                    # 2-digit number
    r')?',                        # End of optional named group *RelativeDoseLevel*
    r')?'                       # End of the optional non-capturing RelativeDose group
    ])

regex_sections.append(relative_dose_pat)

### Combined Targets
- `Total` or `all` indicating that it is the combined structure for all targets 
- If it follows a dose level it indicates that it is the combined structure for 
   all targets at that dose level.

In [21]:
# combined target structures indicated by the word 'Total' or 'All'.
combined_target_def = {
    'Total': 'Combined target structure for all targets at the same dose level',
    'All': 'Combined target structure for all targets at the same dose level',
    'BOTH': 'Combined target structure for all targets at the same dose level'
    }

# Add the combined target definitions to the group_definitions dictionary.
group_definitions['Combined'] = combined_target_def

# Regular expression pattern to match the combined target structures indicator.
combined_targets = '|'.join(target_motion_def.keys())
combined_target_pat = ''.join([
    r'[ _]*',               # Optional space or '_'
    r'(?P<Combined>',       # Start of optional named group *Combined*
    f'{combined_targets}',      # contains '|' separated items from combined_targets.
    r')?',                  # End of optional named group *Combined*
    ])

regex_sections.append(combined_target_pat)


### Target Organ
An organ name following the target type indicates that the target is for a 
specific organ. e.g CTV_Vagina.  

The contents of the organ name has no restrictions except that it must be only 
letters and must be 3 characters or longer.

In [22]:
# Regular expression pattern to match the combined target structures indicator.
target_organ_pat = ''.join([
    r'[ _]*',             # Optional space or '_'
    r'(?P<TargetOrgan>',  # Start of optional named group *TargetOrgan*
    r'[A-Za-z]{{3,}}'         # Name of an organ. (Must be 3 letters or longer.)
    r')?',                # End of optional named group *TargetOrgan*
    ])

regex_sections.append(target_organ_pat)

### Target Expansion
PTVs are sometimes expanded to produce structured used for optimization or to 
evaluate the conformality of a plan. There are three options for the expansion 
volume indicator:
1. A number followed by units (mm or cm)
2. A '+' followed by a number, with the units being optional

While not mentioned in the text, the TG263 examples suggest a third format:

3. `Ev##` (where `##` is the uniform expansion size in mm)


In [23]:
# The three optional expansion patterns are defined separately and them merges with an OR.

# First option: number followed by units
target_expansion_pat_1 = ''.join([
    r'(?P<ExpansionSize1>'  # Start of the *ExpansionSize* group
    r'[0-9.]+'               # Number (one or more digits including decimal)
    r')'                   # End of the *ExpansionSize* group
    r'[ _]*',              # Optional space or '_'
    r'(?P<ExpansionUnit1>'  # Start of the *ExpansionUnit* group
    r'[cm]m'                 # *mm* or *cm*
    r')'                   # End of the *ExpansionUnit* group
    ])

# Second option: '+' followed by number with optional units
target_expansion_pat_2 = ''.join([
    r'[+]'                 # Plus sign
    r' *'                  # Optional white space.
    r'(?P<ExpansionSize2>'  # Start of the *ExpansionSize* group
    r'[0-9.]+'               # Number (one or more digits including decimal)
    r')'                   # End of the *ExpansionSize* group
    r'[ _]*',              # Optional space or '_'
    r'(?P<ExpansionUnit2>'  # Start of the optional *ExpansionUnit* group
    r'[cm]m'                 # *mm* or *cm*
    r')?'                  # End of the optional *ExpansionUnit* group
    ])

# Third option: 'Ev' followed by number with optional units
target_expansion_pat_3 = ''.join([
    r'Ev'                  # Letters 'Ev'
    r'[ _]*',              # Optional space or '_'
    r'(?P<ExpansionSize3>'  # Start of the *ExpansionSize* group
    r'[0-9.]+'               # Number (one or more digits including decimal)
    r')'                   # End of the *ExpansionSize* group
    r'[ _]*',              # Optional space or '_'
    r'(?P<ExpansionUnit3>'  # Start of the optional *ExpansionUnit* group
    r'[cm]m'                 # *mm* or *cm*
    r')?'                  # End of the optional *ExpansionUnit* group
    ])

target_expansion_pat = ''.join([
    r'(?:'     # Start of an optional non-capturing group.
    r'[ _]*',  # Optional space or '_'

    # Merging of the three expansion patterns with an OR.
    '|'.join([target_expansion_pat_1,
              target_expansion_pat_2,
              target_expansion_pat_3]),

    r')?'      # End of the optional non-capturing group
    ])

regex_sections.append(target_expansion_pat)

### Target Organ Subtraction
There are times when modified targets are generated by subtracting a critical 
OAR from the initial target volume
- `-<Organ>`  (Where `<Organ>` represents and valid OAR structure name)

In [24]:
# Regular expression pattern for organ subtraction.
target_minus_organ_pat = ''.join([
    r'(?:'                    # Start of an optional non-capturing group.
    r'[ _]*',                   # Optional space or '_'
    r'-'                        # Minus sign ('-')
    r' *'                       # Optional white space.
    r'(?P<OrganSubtraction>'    # Start of the *OrganSubtraction* group
    r'[A-Za-z]*'                  # OAR label without spaces or numbers.
    r')'                        # End of the *OrganSubtraction* group.
    r')?'                     # End of the optional non-capturing group.
    ])

regex_sections.append(target_minus_organ_pat)

### Target Laterality
- There are times when seperate targets are generated by laterality or other 
   geometrical direction.  Left and right are the most common directions 
   indicators, but the others are also used. Possible directions are:
    - RT, R, LT, L
    - ANT, POST
    - SUP INF
    - MED LAT

- More than one direction indicator can be combined, but not two from the same 
  group. e.g. IGTV_INF, GTV RT, PTV Cavity L, PTV ant rt, opt PTV lat ant



In [25]:
laterality_def = {
    'RT': 'Right',
    'R': 'Right',
    'LT': 'Left',
    'L': 'Left',
    'MED': 'Medial',
    'LAT': 'Lateral'
    }
ap_direction_def = {
    'ANT': 'Medial',
    'POST': 'Lateral'
    }
si_direction_def = {
    'SUP': 'Medial',
    'INF': 'Lateral'
    }

# Add the direction definitions to the group_definitions dictionary.
group_definitions['TargetLaterality'] = laterality_def
group_definitions['TargetAP_Direction'] = ap_direction_def
group_definitions['TargetSI_Direction'] = si_direction_def

# Combine the direction definitions into optional patters separated by '|' (OR).
laterality_types = '|'.join(laterality_def.keys())
ap_types = '|'.join(ap_direction_def.keys())
si_types = '|'.join(si_direction_def.keys())

# Regular expression pattern for direction indicators.
target_direction_pat = ''.join([
    r'[ _]*',                   # Optional space or '_'
    r'(?P<TargetLaterality>'    # Start of the optional *TargetLaterality* group
    r'{laterality_types}'           # contains '|' separated items from laterality_types.
    r')?'                       # End of the optional *TargetLaterality* group.
    r'[ _]*',                   # Optional space or '_'
    r'(?P<TargetAP_Direction>'  # Start of the optional *TargetAP_Direction* group
    r'{ap_types}'                   # contains '|' separated items from ap_types.
    r')?'                       # End of the optional *TargetAP_Direction* group.
    r'[ _]*',                   # Optional space or '_'
   r'(?P<TargetSI_Direction>'   # Start of the optional *TargetSI_Direction* group
    r'{si_types}'                   # contains '|' separated items from si_types.
    r')?'                       # End of the optional *TargetSI_Direction* group.
    ])

regex_sections.append(target_direction_pat)

### Target Subgroup
For optimization purposes a target volume may be divided into subsections 
based on the proximity to hight dose targets.  The TG263 document implies that 
such structures should be prefixed with a `z` or an `_`, but gives no examples.

An alternative approach that would be consistant with TG263 would be to append 
'~' followed by a letter suffix to the full target volume to designate that it 
is a subsection.  
The `~` delimiter would have the same meaning of *partial structure* that it 
has for OAR structures.

e.g., PTV_5040~a, PTV_5040~b, PTV_5040~c

The prefixed with a `z` or an `_` would cause the structure to be ignored by 
any analysis process, so we will ignore that option here as well.
There is no evidence of the ~a, ~b, ~c suffixes being used in clinical practice, 
but it is a reasonable approach, so we will include it here.

The most common practice observed is simply to append a letter suffix to the 
full target volume to designate that it is a subsection.

e.g., PTV_5040a, opt PTV 56 L a

In [26]:
# Regular expression pattern to match the target subgroup indicator.
target_subgroup_pat = ''.join([
    r'[ _]*',                # Optional space or '_'
    r'(?P<TargetSubGroup>',  # Start of optional named group *TargetSubGroup*
    r'[abc]'                     # One of 'a', 'b', or 'c'
    r')?',                   # End of optional named group *TargetSubGroup*
    ])

regex_sections.append(target_subgroup_pat)

### External Target Cropping
- If the structure is cropped back from the external contour for the patient, 
  then the quantity of cropping by “-xx” millimeters is placed at the end of 
  the target string. The cropping length follows the dose indicator, with the 
  amount of cropping indicated by xx millimeters 
  (e.g., PTV_Eval_7000-08, PTV-03, CTVp2-05).

In [27]:
target_crop_pat = ''.join([
    r'(?P<ExternalCrop>',  # Start of optional named group *ExternalCrop*
    r'[0-9]{1,2}',          # 1 or 2 digit Number
    r')?'                  # End of optional named group *ExternalCrop*
    ])

regex_sections.append(target_crop_pat)

In [28]:
# Remaining Text
remainder_pat = ''.join([
    r'(?P<Remainder>'  # Start of the optional *Remainder* group
    r'.+'              # All remaining non-captured text to the end of the string.
    r')?'              # End of the optional *Remainder* group
    r'$'               # End of the string.
    ])

regex_sections.append(remainder_pat)

# Done To Here

In [29]:
final_regex = re.compile(''.join(regex_sections))
matched_groups = []
for structure in include_structures:
    match = final_regex.match(structure)
    if match:
        groups = match.groupdict()
        groups['Structure'] = structure
        matched_groups.append(groups)
    else:
        pprint(f"Structure: {structure} did not match the regex.")
matched_groups_df = pd.DataFrame(matched_groups)

'Structure: BODY did not match the regex.'
'Structure: DPV did not match the regex.'
'Structure: Avoid Post did not match the regex.'
'Structure: Brain did not match the regex.'
'Structure: BrainStem did not match the regex.'
'Structure: Cochlea L did not match the regex.'
'Structure: Cochlea R did not match the regex.'
'Structure: Esophagus did not match the regex.'
'Structure: Globe L did not match the regex.'
'Structure: Globe R did not match the regex.'
'Structure: Larynx did not match the regex.'
'Structure: Lens L did not match the regex.'
'Structure: Lens R did not match the regex.'
'Structure: Mandible did not match the regex.'
'Structure: opt Larynx did not match the regex.'
'Structure: opt Parotid L did not match the regex.'
'Structure: opt Parotid R did not match the regex.'
'Structure: OpticChiasm did not match the regex.'
'Structure: OpticNerve L did not match the regex.'
'Structure: OpticNerve R did not match the regex.'
'Structure: Parotid B did not match the regex.'
'St

In [30]:
matched_groups_df

,Mod,TargetType,Classifier,TargetNumber,Modality,ModalityNumber,Motion,MotionPhase,Nodes,NodalSubSection,...,ExpansionSize3,ExpansionUnit3,OrganSubtraction,TargetLaterality,TargetAP_Direction,TargetSI_Direction,TargetSubGroup,ExternalCrop,Remainder,Structure
0,NaN,CTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,L,CTV 56 L
1,NaN,CTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,R,CTV 56 R
2,NaN,CTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,NaN,CTV 70
3,eval,PTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,NaN,eval PTV 56
4,eval,PTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,NaN,eval PTV 70
5,NaN,GTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,NaN,GTV
6,NaN,GTV,n,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,NaN,GTVn
7,opt,PTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,L a,opt PTV 56 L a
8,opt,PTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,L b,opt PTV 56 L b
9,opt,PTV,NaN,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,R a,opt PTV 56 R a


### Get Dose Values

### Identify match failures

### Utility Functions

In [31]:
def combine_columns(df: pd.DataFrame, columns: List[str], sep=' ')->pd.Series:
    '''Combine text from multiple columns with a separator.

    Args:
        df (pd.DataFrame): The table containing the columns to be merged.
            All of the columns should contain strings or NA values.
        columns (List[str]: The names of the columns to be merged.
        sep (str): A delimiter to place between the text from each column.

    Returns:
        pd.Series: A new text column containing the combined text from each
            column.
    '''
    row_dict = {}
    for index, row in df.iterrows():
        text_items = []
        for col in columns:
            text = row.at[col]
            if text:
                text_items.append(str(text))
        combined_text = sep.join(text_items)
        row_dict[index] = combined_text
    new_col = pd.Series(row_dict)
    return new_col


In [32]:
def to_cgy(text: str)->Dict[str, Union[float, int]]:
    '''Convert numbers with Gy units to cGy and identify fractions.

    Numbers without units or x are assumed to be total dose in cGy.
    Numbers with trailing 'Gy' are converted to cGy.
    Number preceded by x are fractions. dose values are assumed to be dose
    per fraction. Total dose is calculated by:
        $dose_per_fraction x fractions$
    The decimal point may also be represented by a 'p' e.g. 50p4Gy
    If the text does not match any of the valid formats, return the original
    text.

    Args:
        text (str): Dose as a string in one of the following forms:
            ####
            ##Gy
            ##.##Gy
            ##p##Gy
            ####x#
            ##Gyx#
            ##.##Gyx#
            ##p##Gyx#

    Returns:
        Tuple[float, int]: _description_
    '''
    if not text:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Convert 'p' to decimal point.
    try:
        text_cnv1 = text.replace('p', '.')
    except AttributeError:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Find fractions
    dose_parts = text_cnv1.split('x', 1)
    if len(dose_parts) > 1:
        try:
            fractions = int(dose_parts[1])
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    else:
        fractions = None
    dose_str = dose_parts[0]
    # Convert Gy to cGy
    if dose_str.endswith('Gy'):
        try:
            dose = float(dose_str[:-2])  # Drop the Gy suffix
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
        dose = dose * 100   # Gy to cGy conversion
    else:
        try:
            dose = float(dose_str)
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    # Convert dose per fraction to total dose
    if fractions:
        total_dose = dose * fractions
    else:
        total_dose = dose
    dose_dict = {'TotalDose': total_dose, 'Fractions': fractions}
    return pd.Series(dose_dict)


In [33]:
def extract_name_group(names: pd.DataFrame, re_pattern: re.Pattern,
                       match_column: str, idx: pd.Series = None)->pd.DataFrame:
    '''Extract portions of a structure name.

    The re_pattern is applied to the 'Remainder' column to extract named parts.
    The resulting DataFrame is merged with names and the Remainder columns is
    updated with new Remainders from the extraction.

    Args:
        nt_names (pd.DataFrame): A table with structure names. It must contain
            a column 'Remainder', which is used as the starting point for
            extracting name parts.
        re_pattern (re.Pattern): A regular expression with named groups.  It
            must contain one named group that will always be present if a
            successful match is made.  It must also contain a 'Remainder'
            named group that contains the unmatched part of the structure name.
        match_column (str): The name of the named group that is always present
            when a successful match is made.  Used to update the 'Remainder'
            column.
        idx (pd.Series, optional): A mask type index to a subset of names to
            apply the match to.  If not supplied, all names are matched.
            Default is None.

    Returns:
        pd.DataFrame: The supplied table with new columns containing the
            structure name parts.
    '''
    # Extract group parts based on regular expression.
    if idx is not None:
        extr_names = names.loc[idx, 'Remainder'].str.extract(re_pattern)
    else:
        extr_names = names.loc[:, 'Remainder'].str.extract(re_pattern)

    # Merge extracted group parts with structure names.
    names = names.merge(extr_names, how='left',
                            left_index=True, right_index=True,
                            suffixes=('', '_ex'))

    # Update Remainder text
    nt_idx = names[match_column].isna()
    # Where a match was not found, keep the original Remainder text, otherwise
    # update Remainder with resulting Remainder after the match.
    names.Remainder = names.Remainder.where(nt_idx, names.Remainder_ex)
    names.drop(columns=['Remainder_ex'], inplace=True)
    return names


**Structure Set ROI Sequence (3006,0020)**

- **ROI Number (3006,0022)**
    Identification number of the ROI. The value of *ROI Number (3006,0022)* 
    shall be unique within the Structure Set in which it is created.

- **ROI Name (3006,0026)**
    User-defined name for ROI.


**RT ROI Observations Sequence (3006,0080)**

- **Referenced ROI Number (3006,0084)**
    Uniquely identifies the referenced ROI described in the 
    *Structure Set ROI Sequence (3006,0020)*.

- **ROI Observation Label (3006,0085)**
    User-defined label for ROI Observation.

- **RT ROI Interpreted Type (3006,00A4)**
    Type of ROI.
  - Defined Terms:

|Term|Definition|
|----|----------|
|EXTERNAL|external patient contour|
|PTV|Planning Target Volume (as defined in ICRU50)|
|CTV|Clinical Target Volume (as defined in ICRU50)|
|GTV|Gross Tumor Volume (as defined in ICRU50)|
|TREATED_VOLUME|Treated Volume (as defined in ICRU50)|
|IRRAD_VOLUME|Irradiated Volume (as defined in ICRU50)|
|BOLUS|patient bolus to be used for external beam therapy|
|AVOIDANCE|region in which dose is to be minimized|
|ORGAN|patient organ|
|MARKER|patient marker or marker on a localizer|
|REGISTRATION|registration ROI|
|ISOCENTER|treatment isocenter to be used for external beam therapy|
|CONTRAST_AGENT|volume into which a contrast agent has been injected|
|CAVITY|patient anatomical cavity|
|BRACHY_CHANNEL|brachytherapy channel|
|BRACHY_ACCESSORY|brachytherapy accessory device|
|BRACHY_SRC_APP|brachytherapy source applicator|
|BRACHY_CHNL_SHLD|brachytherapy channel shield|
|SUPPORT|external patient support device|
|FIXATION|external patient fixation or immobilization device|
|DOSE_REGION|ROI to be used as a dose reference|
|CONTROL|ROI to be used in control of dose optimization and calculation|